In [36]:
from utils.huggingface import suggest_automodel_class
import torch
from ures.string import format_memory
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

# Transformers Model

In [27]:
model_name = "facebook/opt-125m"
model_class = suggest_automodel_class(model_name)
model = model_class.from_pretrained(model_name)

In [28]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)


In [ ]:
batch_data = next(iter(dataloader))
batch_data = {k: v for k, v in batch_data.items()}

In [37]:
import torchinfo
result = torchinfo.summary(model, input_data=batch_data, col_names=["input_size", "output_size", "num_params"])
print(f"Total Parameters: {format_memory(result.total_param_bytes)}, Activation Size: {format_memory(result.total_output_bytes)}, Memory Size: {format_memory(result.total_input)}")

Total Parameters: 625.03 MB, Activation Size: 1.47 GB, Memory Size: 30.21 KB


# CNN Model

In [45]:
from perf_estimator.models import AllModels
from perf_estimator.dataset import image_dataset

In [47]:
cnn_model = AllModels["ResNet50"].value
cnn_dl = image_dataset()
cnn_input, cnn_out = next(iter(cnn_dl))

In [51]:
import torchinfo
result = torchinfo.summary(cnn_model, mode='train', input_data=cnn_input)
print(f"Total Parameters: {format_memory(result.total_param_bytes)}, Activation Size: {format_memory(result.total_output_bytes)}, Memory Size: {format_memory(result.total_input)}")
result


Total Parameters: 97.49 MB, Activation Size: 5.32 GB, Memory Size: 16.93 MB


Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [200, 1000]               --
├─Conv2d: 1-1                            [200, 64, 43, 43]         9,408
├─BatchNorm2d: 1-2                       [200, 64, 43, 43]         128
├─ReLU: 1-3                              [200, 64, 43, 43]         --
├─MaxPool2d: 1-4                         [200, 64, 22, 22]         --
├─Sequential: 1-5                        [200, 256, 22, 22]        --
│    └─Bottleneck: 2-1                   [200, 256, 22, 22]        --
│    │    └─Conv2d: 3-1                  [200, 64, 22, 22]         4,096
│    │    └─BatchNorm2d: 3-2             [200, 64, 22, 22]         128
│    │    └─ReLU: 3-3                    [200, 64, 22, 22]         --
│    │    └─Conv2d: 3-4                  [200, 64, 22, 22]         36,864
│    │    └─BatchNorm2d: 3-5             [200, 64, 22, 22]         128
│    │    └─ReLU: 3-6                    [200, 64, 22, 22]         --
│ 

# Estimation

In [29]:
from exp.baselines.schedtune import ScheduleTune
cnn_sched = ScheduleTune(
    model=model,
    dataloader=dataloader,
    optimizer=torch.optim.AdamW,
    is_transformer=True,
    device_id=0,
)
print(cnn_sched.parameter_size/1024**3)

# cnn_sched.estimate()
# print(f"Estimated memory: {cnn_sched.estimate_memory} bytes, and execute time: {cnn_sched.execute_time} ns")

0.466552734375
